# Shared address identification

Data exploration aimed at manually identifying ownership groupings for major landowners by looking for related owner names with shared mailing addresses. The results from this analysis have been manually added to `owner-name-groupings.json`.

In [74]:
import json
import pandas as pd
import geopandas as gpd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

# Short display calls
cols = ['PARCELID','CountyName','OwnerName','OwnerAddre','OwnerAdd_1','OwnerAdd_2','OwnerCity','OwnerState','DbaName','CareOfTaxp','joined_address','TotalAcres']

In [ ]:
# Parcel data from **March 9** download -- may need to update file path
parcels = gpd.read_file('./raw/MontanaCadastral_SHP_3-9-2026/Montana_Cadastral/OWNERPARCEL.shp').to_crs(epsg=4326)

In [16]:
# combine various owner address fields in raw data into a single field
parcels['joined_address'] = \
    parcels['OwnerAddre'].astype(str) + ' '\
    + parcels['OwnerAdd_1'].astype(str) + ' '\
    + parcels['OwnerAdd_2'].astype(str) + ' '\
    + parcels['OwnerCity'].astype(str) + ' '\
    + parcels['OwnerState'].astype(str)

In [53]:
# Filter out known public landowners to make the rest of this easier
with open('./known-public-landowners.json', 'r') as file:
    PUBLIC_LANDOWNERS = json.load(file)

# Because the manually compiled public landowners list skips small public landowners, this isn't a perfect "private only date" 
# But it does filter out big ones like the US Forest Service, which makes the rest of this easier

private_parcels = parcels[~parcels['OwnerName'].isin(PUBLIC_LANDOWNERS)].copy()

In [54]:
# Group private parcels by joined_address and OwnerName, then sort by acreage associated with each address
grouped = private_parcels.groupby(['joined_address','OwnerName',]).agg({'PARCELID': 'count','TotalAcres': 'sum'})
by_joined_address = grouped.reset_index().groupby('joined_address').agg({
    "OwnerName": 'unique',
    "PARCELID": 'sum',
    "TotalAcres": 'sum'
}).sort_values('TotalAcres',ascending=False)
by_joined_address.head(30)

,OwnerName,PARCELID,TotalAcres
joined_address,,,
40 71 RANCH LN None None MARTINSDALE MT,"[71 RANCH LP, GALT ERROL T, GALT ERROL T & SHA...",632,252350.384
PO BOX 511196 None None SALT LAKE CITY UT,"[FARMLAND RESERVE INC, FARMLAND RESERVE, INC, ...",464,240014.490
PO BOX 11350 None None BOZEMAN MT,"[BROKEN O LAND & LIVESTOCK LLC, FORT PEASE COM...",551,233217.942
ATTN: PROPERTY TAX DEPT PO BOX 30825 None SALT LAKE CITY UT,"[SUNLIGHT RANCH COMPANY, SUNLIGHT RANCH COMPAN...",686,225446.679
PO BOX 1032 None None CISCO TX,"[WILKS DAN, WILKS FARRIS, WILKS JOANN, WILKS R...",541,204137.205
1301 5TH AVE STE 2700 None None SEATTLE WA,[GREEN DIAMOND RESOURCE COMPANY],442,196071.404
PO BOX 908 None None BOZEMAN MT,"[5 MOUNTAIN RANCH LLC, AMERICAN PRAIRIE FOUNDA...",601,144194.704
MAIL TO: COFFEE CAREN PO BOX 575 None MILES CITY MT,"[COFFEE NEFSY LIMITED PARTNERSHIP, COFFEE RANC...",293,140547.028
9400 SW BARNES RD STE 530 None None PORTLAND OR,"[STIMSOM LUMBER COMPANY, STIMSON LUMBER, STIMS...",388,128556.561


In [55]:
# Display first 30 addresses as JSON to make it easier to see individual Owner Names
print(by_joined_address.reset_index().head(30).to_json(orient='records', indent=4, index=False))

[
    {
        "joined_address":"40 71 RANCH LN None None MARTINSDALE MT",
        "OwnerName":[
            "71 RANCH LP",
            "GALT ERROL T",
            "GALT ERROL T & SHARRIE K",
            "SUN COULEE LLC"
        ],
        "PARCELID":632,
        "TotalAcres":252350.384
    },
    {
        "joined_address":"PO BOX 511196 None None SALT LAKE CITY UT",
        "OwnerName":[
            "FARMLAND RESERVE INC",
            "FARMLAND RESERVE, INC",
            "FARMLAND RESERVES INC"
        ],
        "PARCELID":464,
        "TotalAcres":240014.49
    },
    {
        "joined_address":"PO BOX 11350 None None BOZEMAN MT",
        "OwnerName":[
            "BROKEN O LAND & LIVESTOCK LLC",
            "FORT PEASE COMM PASTURE",
            "FORT PEASE COMMUNITY PASTURE INC",
            "FORT PEASE COMMUNITY PASTURE, INC",
            "KROENKE LAND & LIVESTOCK",
            "KROENKE LAND & LIVESTOCK CO LLC",
            "KROENKE LAND & LIVESTOCK COMPANY LLC",
            "K

As a secondary analysis, I've looked for clusters across similar but not exactly identical addresses as well — this has been a bit of a feel thing.

In [101]:
def search_for_partial_text_in_joined_address(text):
    matches = private_parcels[private_parcels['joined_address'].str.contains(text)][cols]
    print('### SEARCH FOR: "'+ text + '"')
    print('Parcel Count', len(matches))
    print('Acreage', sum(matches['TotalAcres']))
    print('Addresses', matches['joined_address'].unique())
    print('OwnerNames', matches['OwnerName'].unique())
    print('\n')

search_for_partial_text_in_joined_address('N5B CAPITAL')
search_for_partial_text_in_joined_address('LUCERNE CO')
search_for_partial_text_in_joined_address('CISCO TX') # Some but not all of these are Wilks

### SEARCH FOR: "N5B CAPITAL"
Parcel Count 324
Acreage 127171.61
Addresses ['MAIL TO: N5B CAPITAL 601 STATE ST STE 300 None SOUTHLAKE TX'
 'ATTN: N5B CAPITAL 601 STATE ST STE 300 None SOUTHLAKE TX'
 'N5B CAPITAL 601 STATE ST STE 300 None SOUTHLAKE TX']
OwnerNames ['FLATHEAD RIDGE RANCH LLC' 'FRR HUBBARD LLC' 'FRR MURR PEAK LLC'
 'FRR BROWNS MEADOW LLC' 'FRR LOST PRAIRIE LLC' 'FRR SICKLER CREEK LLC'
 'FRR RODGERS LAKE LLC' 'LOOKOUT RIDGE LLC']


### SEARCH FOR: "LUCERNE CO"
Parcel Count 148
Acreage 67031.231
Addresses ['PO BOX 72 None None LUCERNE CO']
OwnerNames ['BOOTH LAND & LIVESTOCK' 'BOOTH LAND & LIVESTOCK CO'
 'BLC DEVELOPMENT LLC' 'BOOTH LAND & LIVESTOCK LLC'
 'INTERMOUNTAIN WEST LIVESTOCK LLC']


### SEARCH FOR: "CISCO TX"
Parcel Count 709
Acreage 275778.759
Addresses ['PO BOX 1032 None None CISCO TX' 'PO BOX 111 None None CISCO TX'
 'PO BOX 984 None None CISCO TX' 'PO BOX 245 None None CISCO TX'
 'ATTN:  JESSICA SULLIVAN PO BOX 111 None CISCO TX'
 '1008 W 11TH ST None None CIS